In [ ]:
# importing needed modules/libraries
!pip install nltk sentence_transformers transformers
import nltk
import numpy as np
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score
from sentence_transformers import SentenceTransformer
from transformers import BertModel, BertTokenizer

nltk.download('treebank')
from nltk.corpus import treebank

In [ ]:
# each sentence is a list of (word, pos_tag) tuples
sentences = treebank.tagged_sents()

# Display first sentence as an example
first_sentence = sentences[0]
print(first_sentence)

# separate the words and the corresponding POS tags
words = []
pos_order = []
for word, pos in first_sentence:
    words.append(word)
    pos_order.append(pos)

# printing sentence and order of POS tags -->  visualization of how the data looks like
print("Sentence:", " ".join(words))
print("POS Order:", " ".join(pos_order))

In [ ]:
# #creating (words, labels) pairs from the treebank
# data = []
# for sentence in sentences:
#     tokens, labels = [], []
#     for word, pos in sentence:
#         tokens.append(word)
#         # logic behind this: if the POS tag starts with "V" (any verb) then label as 1, else 0.
#         label = 1 if pos.startswith("V") else 0
#         labels.append(label)
#     data.append((tokens, labels))

# #"data" is a list of (token_list, label_list) tuples.

# # printing to visualize structure, and to check if something is off
# max_matches = 5  # define how many verb matches to to print
# match_count = 0

# for tokens, labels in data[:5]:
#     for word, label in zip(tokens, labels):
#         if label == 1:  # check if the word is classified as a verb
#             print(f'"{word}" : "verb"')
#             match_count += 1
#             if match_count >= max_matches:
#                 break
#     if match_count >= max_matches:
        
#         break


sentence_texts = []
sentence_labels = []

for sentence in sentences:
    # Extract the words from the sentence
    sentence_words = [word for word, _ in sentence]
    sentence_text = " ".join(sentence_words)
    
    # Check if the sentence contains a verb (any POS tag starting with V)
    contains_verb = 0
    for _, pos in sentence:
        if pos.startswith('V'):
            contains_verb = 1
            break
    
    sentence_texts.append(sentence_text)
    sentence_labels.append(contains_verb)

# Print examples of sentences and their labels
print("\nSentence Examples with Labels:")
for i in range(5):
    print(f"Sentence: {sentence_texts[i]}")
    print(f"Contains verb: {'Yes' if sentence_labels[i] == 1 else 'No'}")
    print()

In [ ]:
# Using SentenceTransformer for sentence embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
sentence_embeddings = model.encode(sentence_texts)

print(f"Number of sentences: {len(sentence_texts)}")
print(f"Shape of embeddings: {sentence_embeddings.shape}")

In [ ]:
def get_bert_sentence_embedding(sentence):
    """
    Given a sentence, return the BERT-based sentence embedding.
    This implementation uses the [CLS] token's representation as the sentence embedding.
    """
    bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    bert_model = BertModel.from_pretrained('bert-base-uncased')
    
    # Tokenize sentence with appropriate padding and truncation
    inputs = bert_tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
    
    # Disable gradients, only doing inference --> no training.
    with torch.no_grad():
        outputs = bert_model(**inputs)
    
    # outputs.last_hidden_state has shape (batch_size, sequence_length, hidden_size)
    # Embedding for the [CLS] token is at position 0 of the sequence.
    cls_embedding = outputs.last_hidden_state[:, 0, :]
    
    return cls_embedding.numpy()

In [ ]:
def train_logistic_regression(features, labels):
    """
    Train logistic regression model on sentence features.
    """
    # splitting into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, test_size=0.2, random_state=42
    )
    
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)
    
    # predictions on the test set
    predictions = clf.predict(X_test)
    
    # evaluation
    metrics = {
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions),
        "recall": recall_score(y_test, predictions),
        "f1_score": f1_score(y_test, predictions)
    }
    
    return clf, metrics


if __name__ == "__main__":
    # Use BERT embeddings for each sentence
    X_features = []
    for sentence in sentence_texts:
        tokens = sentence.split()
        embedding = get_bert_sentence_embedding(tokens)
        X_features.append(embedding.flatten())
    
    X_features = np.array(X_features)
    
    # Train logistic regression on the BERT sentence embeddings
    clf, eval_metrics = train_logistic_regression(X_features, np.array(sentence_labels))
    print("Evaluation Metrics on BERT sentence embeddings:")
    print(eval_metrics)

In [ ]:
def predict_sentence_contains_verb(sentence, model, clf):
    # Encode the sentence
    embedding = model.encode([sentence])[0].reshape(1, -1)
    
    # Predict using the classifier
    prediction = clf.predict(embedding)[0]
    probability = clf.predict_proba(embedding)[0][1]  # Probability of class 1
    
    return prediction, probability

# Test examples
test_sentences = [
    "The cat is sleeping on the couch.",  # Contains verb
    "A beautiful sunset over the mountains.",  # No verb
    "She runs every morning before breakfast.",  # Contains verb
    "The big red book.",  # No verb
    "The big man" #no verb
]

print("\nPredictions on test sentences:")
for sentence in test_sentences:
    pred, prob = predict_sentence_contains_verb(sentence, model, clf)
    print(f"Sentence: {sentence}")
    print(f"Contains verb: {'Yes' if pred == 1 else 'No'} (probability: {prob:.4f})")
    print()